# Анализ лояльности пользователей Яндекс Афиши

## Этапы выполнения проекта

### 1. Загрузка данных и их предобработка

---

**Задача 1.1:** Напишите SQL-запрос, выгружающий в датафрейм pandas необходимые данные. Используйте следующие параметры для подключения к базе данных `data-analyst-afisha`:

Для выгрузки используйте запрос из предыдущего урока и библиотеку SQLAlchemy.

Выгрузка из базы данных SQL должна позволить собрать следующие данные:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа (используйте данные `created_dt_msk`);
- `order_ts` — дата и время создания заказа (используйте данные `created_ts_msk`);
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

---


In [1]:
# Используйте ячейки типа Code для вашего кода,
# а ячейки типа Markdown для комментариев и выводов

In [2]:
# При необходимости добавляйте новые ячейки для кода или текста

In [3]:
#установим необходимые библиотеки
#!pip install sqlalchemy

In [4]:
#!pip install psycopg2-binary

In [5]:
#установим библиотеку phik
#!pip install phik

In [6]:
#библиотека для загрузки данных из .env
#!pip install python-dotenv

In [7]:
#импортируем библиотеки
import pandas as pd
from sqlalchemy import create_engine

# Загружаем библиотеки для визуализации данных
import matplotlib.pyplot as plt
import seaborn as sns

# Загружаем библиотеку для расчёта коэффициента корреляции phi_k
from phik import phik_matrix

In [8]:
#ниже код для загрузки из .env файла
# Импорт load_dotenv.
from dotenv import load_dotenv 

# Импорт библиотеки для работы с окружением.
import os  

#print(os.getcwd())

# Загрузка переменных из .env
load_dotenv(dotenv_path='./.env')

# Теперь переменные доступны через os.environ
user = os.getenv('USER_DB')
pwd = os.getenv('PWD_DB')
host = os.getenv('HOST_DB')
port = os.getenv('PORT_DB')
db = os.getenv('DB')


In [9]:
#данные для подключения к базе данных
db_config = {'user': user, # имя пользователя
             'pwd': pwd, # пароль
             'host': host,
             'port': port, # порт подключения
             'db':db # название базы данных
             }

In [10]:
#сформируем строку для подключения
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db'],
)

In [11]:
#создадим соединение
engine = create_engine(connection_string)

In [12]:
#напишем запрос
query = '''
SELECT p.user_id,
p.device_type_canonical,
p.created_dt_msk AS order_dt,
p.created_ts_msk AS order_ts,
p.currency_code,
p.revenue,
p.tickets_count,
EXTRACT(DAY FROM created_dt_msk -LAG(created_dt_msk) OVER(PARTITION BY user_id ORDER BY created_dt_msk)) AS days_since_prev,
p.event_id,
e.event_name_code AS event_name,
e.event_type_main,
p.service_name,
r.region_name,
c.city_name
FROM afisha.purchases AS p
JOIN afisha.events AS e USING (event_id)
JOIN afisha.city AS c USING (city_id)
JOIN afisha.regions AS r USING (region_id)
WHERE (device_type_canonical = 'mobile' OR device_type_canonical = 'desktop')
AND e.event_type_main <> 'фильм'
ORDER BY user_id
'''

In [ ]:
#запишем результат запроса в датафрейм
df = pd.read_sql_query(query, con=engine)

---

**Задача 1.2:** Изучите общую информацию о выгруженных данных. Оцените корректность выгрузки и объём полученных данных.

Предположите, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксируйте основную информацию о данных в кратком промежуточном выводе.

---

In [ ]:
#посмотрим общую информацию
df.info()

In [ ]:
#проверим корректность данных
df.head()

Видим, что датафрейм состоит из 12 столбцов и 290611 строк. Типы определились корректно, однако на стадии предобработки данных необходимо обработать пропуски, оптимизировать типы данных.

In [ ]:
# создаем копию датасета до преобразования для возможности проверить сделанные изменения после предобработки
temp = df.copy() 
len(temp)

---

###  2. Предобработка данных

Выполните все стандартные действия по предобработке данных:

---

**Задача 2.1:** Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

Для этого используйте датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохраните в новый столбец `revenue_rub`.

---


In [ ]:
#считаем датасет с информацией о курсе казахстанского тенге
final_tickets_tenge_df = pd.read_csv('https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')

In [ ]:
#посмотрим, что считалось
final_tickets_tenge_df

In [ ]:
#посмотрим информацию о датафрейме
final_tickets_tenge_df.info()

In [ ]:
#приведем столбец с датой к типу datetime
final_tickets_tenge_df['data'] = pd.to_datetime(final_tickets_tenge_df['data'])

In [ ]:
#объединим эти два датафрейма по дате
df_merged = pd.merge(df, final_tickets_tenge_df, left_on='order_dt', right_on='data' )

In [ ]:
#проверим, что никакие строки не пропали
len(df)

In [ ]:
len(df_merged)

In [ ]:
#приведем всю выручку к рублю, округляя до двух знаков после запятой
df.loc[df['currency_code'] == 'rub', 'revenue_rub'] = round(df['revenue'],2)
df.loc[df['currency_code'] == 'kzt', 'revenue_rub'] = round(df['revenue'] / 100 * df_merged['curs'],2)

---

**Задача 2.2:**

- Проверьте данные на пропущенные значения. Если выгрузка из SQL была успешной, то пропуски должны быть только в столбце `days_since_prev`.
- Преобразуйте типы данных в некоторых столбцах, если это необходимо. Обратите внимание на данные с датой и временем, а также на числовые данные, размерность которых можно сократить.
- Изучите значения в ключевых столбцах. Обработайте ошибки, если обнаружите их.
    - Проверьте, какие категории указаны в столбцах с номинальными данными. Есть ли среди категорий такие, что обозначают пропуски в данных или отсутствие информации? Проведите нормализацию данных, если это необходимо.
    - Проверьте распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха.
        
        Важные показатели в рамках поставленной задачи — это выручка с заказа (`revenue_rub`) и количество билетов в заказе (`tickets_count`), поэтому в первую очередь проверьте данные в этих столбцах.
        
        Если обнаружите выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.

После предобработки проверьте, были ли отфильтрованы данные. Если были, то оцените, в каком объёме. Сформулируйте промежуточный вывод, зафиксировав основные действия и описания новых столбцов.

---

Проверим данные на пропущенные значения.

In [ ]:
def show_missing_stats(tmp0):
    """
    Функция для отображения статистики пропущенных значений в DataFrame.
    """
    missing_stats = pd.DataFrame({
        'Кол-во пропусков': tmp0.isnull().sum(),
        'Доля пропусков': tmp0.isnull().mean()
    })
    missing_stats = missing_stats[missing_stats['Кол-во пропусков'] > 0]
    
    if missing_stats.empty:
        return "Пропусков в данных нет"
    
    # Форматируем при выводе через Styler
    return (missing_stats.style.format({'Доля пропусков': '{:.4f}'}).background_gradient(cmap='coolwarm'))
show_missing_stats(df)

Пропуски присутствуют только в столбце `days_since_days`, что и должно быть, так как в этом столбце хранится количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено. Оставим эти пропуски без изменений.

Оптимизируем размерность в типах данных.

In [ ]:
for column in ['revenue','days_since_prev', 'revenue_rub']:
    df[column] = pd.to_numeric(df[column], downcast='float')
    
for column in ['tickets_count','event_id']:
    df[column] = pd.to_numeric(df[column], downcast='integer')

In [ ]:
#проверим оптимизацию
df.info()

In [ ]:
df.head()

Проверим полные дубликаты.

In [ ]:
df.duplicated().sum()

In [ ]:
#удалим эти дубликаты
df = df.drop_duplicates()

Изучим значения в ключевых столбцах.

In [ ]:
# Проверяем уникальные значения в столбцах
for column in ['device_type_canonical', 'currency_code', 'tickets_count', 'days_since_prev', 
               'event_type_main', 'service_name', 'region_name','city_name']:
    print(f'Уникальные значения в столбце {column}:')
    print(df[column].sort_values().unique())
    print()

В категориальных данных все значения уникальны, значений обозначающих пропуски в них нет, нормализация данных не требуется.

Проверим распределение численных данных и наличие в них выбросов. Для этого используем статистические показатели, гистограммы распределения значений или диаграммы размаха.

Важные показатели в рамках поставленной задачи — это выручка с заказа `revenue_rub` и количество билетов в заказе `tickets_count`, поэтому в первую очередь проверим данные в этих столбцах.

In [ ]:
df.head()

In [ ]:
# Изучаем статистические показатели столбца revenue_rub
print('Статистические показатели столбца revenue_rub:')
df['revenue_rub'].describe()

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(10, 4))

# Строим диаграмму размаха значений в столбце revenue_rub
df.boxplot(column='revenue_rub', vert=False)

# Добавляем заголовок и метки оси
plt.title('Распределение выручки в рублях')
plt.xlabel('Выручка в рублях')

# Выводим график
plt.show()

Данные в этом столбце распределены неравномерно, наблюдается правый хвост, среднее значение 555.18 сильно отличается от медианного 351.14. Это происходит из-за выбросов, к таковым в данном случае относятся значения, превыщающие 1835 рублей.

Отфильтруем значения по 99 перцентилю.

In [ ]:
# Найдем 99-й процентиль
outliers = df['revenue_rub'].quantile(0.99)

# Отфильтруем данные, оставив значения меньше найденного порога выбросов
df = df[df['revenue_rub']< outliers]

# Выводим результат describe() после фильтрации данных
df['revenue_rub'].describe()

In [ ]:
#посмотрим сколько данных отфильтровалось
print('Абсолютное значение отфильтрованных двнных',len(temp) - len(df))
print('Относительное значение отфильтрованных двнных',round((len(temp) - len(df))/len(temp),5))

Таким образом, отфильтровали 1% данных.

In [ ]:
# Изучаем статистические показатели столбца tickets_count
print('Статистические показатели столбца tickets_count:')
df['tickets_count'].describe()

In [ ]:
#посмотрим как часто встречается конкретное число купленных билетов
df['tickets_count'].value_counts()

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(10, 4))

# Строим гистограмму с помощью pandas через plot(kind='hist')
df['tickets_count'].plot(
                kind='hist', # Тип графика - гистограмма
                bins=50,# Устанавливаем количество корзин 
                alpha=0.75,
                edgecolor='black'
   
             
)

# Настраиваем оформление графика
plt.title('Распределение количества купленных билетов')
plt.xlabel('Количество купленных билетов')
plt.ylabel('Частота')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(10, 4))

# Строим диаграмму размаха значений в столбце tickets_count
df.boxplot(column='tickets_count', vert=False)

# Добавляем заголовок и метки оси
plt.title('Распределение количества купленных билетов')
plt.xlabel('Количество купленных билетов')

# Выводим график
plt.show()

Данные в этом столбце также распределены неравномерно, наблюдается правый хвост, к выбросам в данном случае относятся значения, превыщающие 7. В среднем покупают 3 билета.

В результате предобработки данных были выполнены следующие действия:

Вся выручка приведена к единой валюте - рублю, округляя до двух знаков после запятой, в соответствии с курсом тенге в конкретную дату. Данные были сохранены в столбце `revenue_rub`.
    
Изучены пропуски в данных. Пропуски присутствуют только в столбце days_since_days, что и должно быть, так как в этом столбце хранится количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено. Пропуски оставлены без изменений.

Изучены типы данных. Оптимизирован тип данных float в столбцах `revenue`,`days_since_prev`, `revenue_rub` и тип данных int в столбцах`tickets_count`,`event_id`.

Изучены ключевые значения в столбцах. В категориальных данных все значения уникальны, значений обозначающих пропуски в них нет, нормализация данных не требуется.

Данные проверили на явные и неявные дубликаты. Было найдено 31 явных дубликатов, они были удалены. Неявные дубликаты отсутствуют.

Проверено распределение численных данных и наличие в них выбросов. Данные в столбце `revenue_rub` распределены неравномерно с правым хвостом, среднее значение 555.18 сильно отличается от медианного 351.14. Это происходит из-за выбросов, к таковым в данном случае относятся значения, превыщающие 1835 рублей. Значения в этом столбце были отфильтрованы по 99 перцентилю.

Данные в столбце `tickets_count` также распределены неравномерно, наблюдается правый хвост, к выбросам в данном случае относятся значения, превыщающие 7. В среднем покупают 3 билета.

---

### 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

---

**Задача 3.1.** Постройте профиль пользователя — для каждого пользователя найдите:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используйте поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавьте два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.

**Рекомендация:** перед тем как строить профиль, отсортируйте данные по времени совершения заказа.

---


In [ ]:
#отсортируем данные по времени совершения заказа
df = df.sort_values(by='order_ts', ascending=True)

In [ ]:
df.head()

In [ ]:
#создадим новый датафрейм с профилями пользователей
profile_df = pd.DataFrame()

#сгруппируем по user_id
grouped = df.groupby('user_id')

#формируем новый датафрейм
profile_df['min_order_dt'] = grouped['order_dt'].min()
profile_df['max_order_dt'] = grouped['order_dt'].max()
profile_df['first_device_type_canonical'] = grouped['device_type_canonical'].first()
profile_df['first_region_name'] = grouped['region_name'].first()
profile_df['first_service_name'] = grouped['service_name'].first()
profile_df['first_event_type_main'] = grouped['event_type_main'].first()
profile_df['count_orders'] = grouped['event_id'].count()
profile_df['avg_revenue_rub'] = grouped['revenue_rub'].mean()
profile_df['avg_tickets_count'] = grouped['tickets_count'].mean()
profile_df['avg_time_between_orders'] = grouped['order_ts'].apply(lambda group: group.sort_values().diff().mean())


In [ ]:
#напишем функцию, которая отмечает True совершил ли пользователь 2 и более заказа
def create_is_two(row):
    if row['count_orders'] >= 2:
           return True
    return False

In [ ]:
#добавим поле с бинарным признаком is_two
profile_df['is_two'] = profile_df.apply(create_is_two, axis=1)

In [ ]:
#напишем функцию, которая отмечает True совершил ли пользователь 5 и более заказов
def create_is_five(row):
    if row['count_orders'] >= 5:
           return True
    return False

In [ ]:
#добавим поле с бинарным признаком is_five
profile_df['is_five'] = profile_df.apply(create_is_five, axis=1)

In [ ]:
#посмотрим результат
profile_df.head(10)

---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными вы работаете: насколько они репрезентативны и нет ли в них аномалий.

Используя данные о профилях пользователей, рассчитайте:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших 2 и более заказа;
- долю пользователей, совершивших 5 и более заказов.

Также изучите статистические показатели:

- по общему числу заказов;
- по среднему числу билетов в заказе;
- по среднему количеству дней между покупками.

По результатам оцените данные: достаточно ли их по объёму, есть ли аномальные значения в данных о количестве заказов и среднем количестве билетов?

Если вы найдёте аномальные значения, опишите их и примите обоснованное решение о том, как с ними поступить:

- Оставить и учитывать их при анализе?
- Отфильтровать данные по какому-то значению, например, по 95-му или 99-му перцентилю?

Если вы проведёте фильтрацию, то вычислите объём отфильтрованных данных и выведите статистические показатели по обновлённому датасету.

In [ ]:
#рассчитаем общее число пользователей
len(profile_df)

In [ ]:
#рассчитаем среднюю выручку с одного заказа
round(profile_df['avg_revenue_rub'].mean(),2)

In [ ]:
#рассчитаем долю пользователей, совершивших 2 и более заказа
round(profile_df['is_two'].mean(),2)

In [ ]:
#рассчитаем долю пользователей, совершивших 5 и более заказа
round(profile_df['is_five'].mean(),2)

In [ ]:
#изучим статистические показатели по общему числу заказов
profile_df['count_orders'].describe()

In [ ]:
#изучим статистические показатели по среднему числу билетов в заказе
profile_df['avg_tickets_count'].describe()

In [ ]:
#изучим статистические показатели по среднему количеству дней между покупками
profile_df['avg_time_between_orders'].describe()

В выборке число пользователей составляет `21844`. Cредняя выручка с одного заказа составляет `542.75` рубля.
Доля пользователей, совершивших 2 и более заказа, составляет `0.62`, то есть почти 2/3 всех пользователей совершили 2 и более заказа. Доля пользователей, совершивших 5 и более заказов, составляет `0.29`, то есть почти 1/3 всех пользователей совершила 5 и более заказов.

Данных достаточно по объему.

Из статистических показателей по общему числу заказов видно, что минимальное число заказов составляет 1, максимальное 10181, среднее 13,1, а медианное 2, то есть данные распределены неравномерно, из-за выбросов искажено среднее значение. Огромные числа, такое как максимальное значение, можно отнести к аномальным значениям, и отфильтровать по 99 процентилю.

Из статистических показателей по среднему числу билетов в заказе видно, что минимальное число билетов в заказе составляет 1, максимальное 11, среднее 2,75 и медианное 2,75 , то есть данные распределены равномерно.

Из статистических показателей по среднему количеству дней между покупками видно, что минимальное количество дней между покупками (если заказ не один, а таких 13482 записи) составляет 1 секунду, максимальное 148 дней, среднее почти 16 дней, а медианное чуть больше 8 дней , что говорит о том, что данные распределены неравномерно. Но большие значения между заказами вполне могут быть в реальности, поэтому их не стоит относить к аномальным.

Отфильтруем значения по 99 процентилю.

In [ ]:
# создаем копию датасета до преобразования для возможности проверить сделанные изменения
temp_profile_df = profile_df.copy() 
len(temp_profile_df)

In [ ]:
# Найдем 99-й процентиль
outliers_count_orders = profile_df['count_orders'].quantile(0.99)

# Отфильтруем данные, оставив значения меньше найденного порога выбросов
profile_df = profile_df[profile_df['count_orders']< outliers_count_orders]

# Выводим результат describe() после фильтрации данных
profile_df['count_orders'].describe()

In [ ]:
#посмотрим сколько данных отфильтровалось
print('Абсолютное значение отфильтрованных данных',len(temp_profile_df) - len(profile_df))
print('Относительное значение отфильтрованных данных',round((len(temp_profile_df) - len(profile_df))/len(temp_profile_df),5))

Таким образом, отфильтровали 1% данных.

---

### 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используйте профили пользователей.



#### 4.1. Исследование признаков первого заказа и их связи с возвращением на платформу

Исследуйте признаки, описывающие первый заказ пользователя, и выясните, влияют ли они на вероятность возвращения пользователя.

---

**Задача 4.1.1.** Изучите распределение пользователей по признакам.

- Сгруппируйте пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа;
    - по билетному оператору, продавшему билеты на первый заказ.
- Подсчитайте общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. Сегмент — это группа пользователей, объединённых определённым признаком, то есть объединённые принадлежностью к категории. Например, все клиенты, сделавшие первый заказ с мобильного телефона, — это сегмент.
- Ответьте на вопрос: равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей?

---


Изучим распределение пользователей по признакам,описывающим первый заказ пользователя. Начнем с типа их первого мероприятия. Для визуализации используем столбчатую диаграмму.

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(10, 4))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
profile_df['first_event_type_main'].value_counts().plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=45, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение пользователей в зависимости от типа первого мероприятия, посещенного ими'
)

# Настраиваем оформление графика
plt.xlabel('Тип первого мероприятия')
plt.ylabel('Количество пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

In [ ]:
#посмотрим на цифры
profile_df['first_event_type_main'].value_counts()

Теперь посмотрим на их долю.

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(10, 4))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
(profile_df['first_event_type_main'].value_counts()/len(profile_df)).plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=45, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение долей пользователей в зависимости от типа первого мероприятия, посещенного ими'
)

# Настраиваем оформление графика
plt.xlabel('Тип первого мероприятия')
plt.ylabel('Доля пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

In [ ]:
profile_df['first_event_type_main'].value_counts()/len(profile_df)

Наибольшая доля пользователей (44%) в качестве первого мероприятия выбирала `концентры`, далее идет категория `другое` (25%) и замыкает тройку `театр` (19,6 %). Остальные категории сильно уступают в качестве первой категории мероприятия.

Изучим распределение по типу устройства, с которого совершена первая покупка.

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(10, 4))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
profile_df['first_device_type_canonical'].value_counts().plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=45, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение пользователей в зависимости от типа устройства, с которого совершена первая покупка'
)

# Настраиваем оформление графика
plt.xlabel('Тип устройства')
plt.ylabel('Количество пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

In [ ]:
#посмотрим на цифры
profile_df['first_device_type_canonical'].value_counts()

Теперь посмотрим на их долю.

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(10, 4))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
(profile_df['first_device_type_canonical'].value_counts()/len(profile_df)).plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=45, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение долей пользователей в зависимости от типа устройства, с которого совершена первая покупка'
)

# Настраиваем оформление графика
plt.xlabel('Тип устройства')
plt.ylabel('Доля пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

In [ ]:
profile_df['first_device_type_canonical'].value_counts()/len(profile_df)

Мобильные телефоны сильно лидируют в качестве типа устройства, с которого пользователи совершают первую покупку, по сравнению с компьютерами, `82,8 %` против `17,1 %`.

Изучим распределение по региону проведения мероприятия из первого заказа.

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(13, 5))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
profile_df['first_region_name'].value_counts().plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=90, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение пользователей в зависимости от региона проведения мероприятия из первого заказа'
)

# Настраиваем оформление графика
plt.xlabel('Регион проведения мероприятия')
plt.ylabel('Количество пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

In [ ]:
#посмотрим на цифры
profile_df['first_region_name'].value_counts()

Теперь посмотрим на их долю.

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(13, 5))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
(profile_df['first_region_name'].value_counts()/len(profile_df)).plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=90, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение долей пользователей в зависимости от региона проведения мероприятия из первого заказа'
)

# Настраиваем оформление графика
plt.xlabel('Регион проведения мероприятия')
plt.ylabel('Доля пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

In [ ]:
profile_df['first_region_name'].value_counts()/len(profile_df)

Каждый третий пользователь делал первый заказ из `Каменевского региона`, далее идет `Североярская область` (17,4%) и `Широковская область` (5.6%). Встречается очень много регионов с долей менее 1 %.

Изучим распределение по билетному оператору, продавшему билеты на первый заказ.

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(10, 4))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
profile_df['first_service_name'].value_counts().plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=90, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение пользователей в зависимости от билетного оператора, продавшего билеты на первый заказ'
)

# Настраиваем оформление графика
plt.xlabel('Билетный оператор')
plt.ylabel('Количество пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

In [ ]:
#посмотрим на цифры
profile_df['first_service_name'].value_counts()

Теперь посмотрим на их долю.

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(13, 5))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
(profile_df['first_service_name'].value_counts()/len(profile_df)).plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=90, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение долей пользователей в зависимости от билетного оператора, продавшего билеты на первый заказ'
)

# Настраиваем оформление графика
plt.xlabel('Билетный оператор')
plt.ylabel('Доля пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

In [ ]:
profile_df['first_service_name'].value_counts()/len(profile_df)

Наиболее популярным билетным оператором среди первых заказов стал оператор `Билеты без проблем` с долей `23,9 %`, далее идет `Мой билет` с долей `13,6%` и замыкает тройку `Лови билет!` с долей `12,9%`. Больше половины оперторов не занимает и 1 %. 

---

**Задача 4.1.2.** Проанализируйте возвраты пользователей:

- Для каждого сегмента вычислите долю пользователей, совершивших два и более заказа.
- Визуализируйте результат подходящим графиком. Если сегментов слишком много, то поместите на график только 10 сегментов с наибольшим количеством пользователей. Такое возможно с сегментами по региону и по билетному оператору.
- Ответьте на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?

При интерпретации результатов учитывайте размер сегментов: если в сегменте мало пользователей (например, десятки), то доли могут быть нестабильными и недостоверными, то есть показывать широкую вариацию значений.

---


Вычислим долю пользователей, совершивших два и более заказа, в сегментах по типу из первого мероприятия и визуализируем с помощью столбчатой диаграммы.

In [ ]:
profile_df.groupby('first_event_type_main')['is_two'].mean().sort_values(ascending=False)

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(13, 5))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
profile_df.groupby('first_event_type_main')['is_two'].mean().sort_values(ascending=False).plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=45, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение долей пользователей, совершивших два и более заказа, в сегментах по типу первого мероприятия'
)

# Настраиваем оформление графика
plt.xlabel('Тип первого мероприятия')
plt.ylabel('Доля пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

В целом показатели находятся примерно на одном уровне. Рассмотрим тройку лидеров. В категории `выставки` 64% пользователей, совершили 2 и более заказа, далее идет категория `театр` (63,4%) и категория `концерты` (61,8%).
Однако в сегменте `выставки` мало пользователей, и доля может быть недостоверной. Про остальные две категории можно сказать, что пользователи в них чаще возвращаются на Яндекс Афишу.

Вычислим долю пользователей, совершивших два и более заказа, в сегментах по типу устройства, с которого совершена первая покупка и визуализируем с помощью столбчатой диаграммы.

In [ ]:
profile_df.groupby('first_device_type_canonical')['is_two'].mean().sort_values(ascending=False)

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(13, 5))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
profile_df.groupby('first_device_type_canonical')['is_two'].mean().sort_values(ascending=False).plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=45, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение долей пользователей, совершивших два и более заказа, в сегментах по типу устройства, с которого совершена первая покупка'
)

# Настраиваем оформление графика
plt.xlabel('Тип устройства')
plt.ylabel('Доля пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

В целом примерно 60% пользователей в каждом сегменте совершают 2 и более заказа. При этом данных в каждом сегменте достаточно, хотя данные в этих сегментах распределены неравномерно (82,8 % пользователей совершали первый заказ с мобильного телефона, а 17,1 % с компьютера).

Вычислим долю пользователей, совершивших два и более заказа, в сегментах по региону проведения мероприятия из первого заказа и визуализируем с помощью столбчатой диаграммы.


In [ ]:
profile_df.groupby('first_region_name')['is_two'].mean().sort_values(ascending=False)

Cегментов слишком много, по заданию поместим на график только 10 сегментов с наибольшим количеством пользователей.

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(13, 5))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
profile_df.groupby('first_region_name')['is_two'].mean().sort_values(ascending=False)[0:10].plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=90, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение долей пользователей, совершивших два и более заказа, в сегментах по региону проведения мероприятия из первого заказа'
)

# Настраиваем оформление графика
plt.xlabel('Регион')
plt.ylabel('Доля пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

В `Верхнеозерском крае` 100% пользователей совершили 2 и более заказов (а это потому что там вообще 1 пользователь). В `Озернопольской области` 89,6 % пользователей (здесь их тоже очень мало),а в `Радужнопольском крае` 78,2% (и здесь их очень мало). 

Пятерки регионов(данные с количеством пользователей ниже), лидирующих по количеству пользователей, здесь нет.

Каменевский регион          7082

Североярская область        3767

Широковская область         1224

Озернинский край             675

Малиновоярский округ         526

Вычислим долю пользователей, совершивших два и более заказа, в сегментах по билетному оператору, продавшему билеты на первый заказ и визуализируем с помощью столбчатой диаграммы.


In [ ]:
profile_df.groupby('first_service_name')['is_two'].mean().sort_values(ascending=False)

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(13, 5))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
profile_df.groupby('first_service_name')['is_two'].mean().sort_values(ascending=False)[0:10].plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=90, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение долей пользователей, совершивших два и более заказа, в сегментах по билетному оператору, продавшему билеты на первый заказ'
)

# Настраиваем оформление графика
plt.xlabel('Билетный оператор')
plt.ylabel('Доля пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

У первых в этом списке достаточно малое количество пользователей, поэтому нельзя считать, что они корректно занимают лидирующие места. В этой десятке можно отметить только операторов `Край билетов` c количеством пользователей 454 и `Дом культуры` с количеством пользователей 353. Доли пользователей, совершивших 2 и более заказов, равны около 65 %.

---

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

---

Проверим 1 гипотезу: Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.

In [ ]:
profile_df.groupby('first_event_type_main')['is_two'].mean().sort_values(ascending=False)

In [ ]:
#посмотрим на количество пользователей в сегментах
profile_df['first_event_type_main'].value_counts()

Видим, что доля пользователей, совершивших 2 и более заказов, у которых первый заказ был на спортивные мероприятия составляет 55,7 %, а у которых первый заказ был на концерт составляет 61,8 %.
При этом стоит отметить, что пользователей, совершивших первый заказ на концерт гораздо больше, чем тех, что относятся к сегменту `спорт`.

Вывод: гипотеза неверна.

Проверим 2 гипотезу: В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

In [ ]:
mean_regions = profile_df.groupby('first_region_name')['is_two'].mean().sort_values(ascending=False)
mean_regions

In [ ]:
#посмотрим на количество пользователей в сегментах
count_regions = profile_df['first_region_name'].value_counts()
count_regions

In [ ]:
merge_profile_df_on_region = pd.merge(mean_regions, count_regions, left_index=True, right_index=True)
merge_profile_df_on_region.sort_values(by='first_region_name', ascending=False)[0:30]

Для тех данных, где количество пользователей не слишком мало, в целом можно сделать вывод, что гипотеза верна. 

У первой тройки по количеству пользователей `Каменевский регион`, `Североярская область` и `Широковская область` доля тех, кто совершил 2 и более заказов, составляет около 63 %, что в целом больше, остальных. Хотя встречаются региона с меньшим количеством пользователей, но с долей равной или немного превышающей это значение.

---

#### 4.2. Исследование поведения пользователей через показатели выручки и состава заказа

Изучите количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.

---

**Задача 4.2.1.** Проследите связь между средней выручкой сервиса с заказа и повторными заказами.

- Постройте сравнительные гистограммы распределения средней выручки с билета (`avg_revenue_rub`):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответьте на вопросы:
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?

Текст на сером фоне:
    
**Рекомендация:**

1. Используйте одинаковые интервалы (`bins`) и прозрачность (`alpha`), чтобы визуально сопоставить распределения.
2. Задайте параметру `density` значение `True`, чтобы сравнивать форму распределений, даже если число пользователей в группах отличается.

---


In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(7, 3))

# Находим минимальное и максимальное значения
min_value = int(profile_df['avg_revenue_rub'].min())
max_value = int(profile_df['avg_revenue_rub'].max())

# Строим гистограммы для каждого значения is_two
for i in profile_df['is_two'].unique():
    # Фильтруем данные по значению столбца is_two
    profile_df.loc[profile_df['is_two'] == i, 'avg_revenue_rub'].plot(
        kind='hist',
        density=True,
        bins=range(min_value, max_value+1, 50),
        alpha=0.5,
        label=f'{i}',
        legend=True
    )

# Настраиваем оформление графика
plt.title('Сравнение распределения средней выручки с билета в зависимости от признака is_two')
plt.xlabel('Выручка с билета')
plt.ylabel('Плотность вероятности')
plt.legend(title='Значение is_two')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()



In [ ]:
for i in profile_df['is_two'].unique():
    # Фильтруем данные по значению столбца is_two
    print(profile_df.loc[profile_df['is_two'] == i, 'avg_revenue_rub'].describe())

На графике видно, что распределение средней выручки у тех, кто совершил 2 и более заказа, сдвинуто в правую область по отношению к тем, кто совершил один заказ: в среднем выручка у пользователей, которые возвращаются 400-600 рублей. А у пользователей совершивших один заказ показатели выручки распределены неравномерно, с правым хвостом, медианное значение 377 р.

---

**Задача 4.2.2.** Сравните распределение по средней выручке с заказа в двух группах пользователей:

- совершившие 2–4 заказа;
- совершившие 5 и более заказов.

Ответьте на вопрос: есть ли различия по значению средней выручки с заказа между пользователями этих двух групп?

---


In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(7, 3))

# Находим минимальное и максимальное значения
min_value = int(profile_df['avg_revenue_rub'].min())
max_value = int(profile_df['avg_revenue_rub'].max())

for i in profile_df['is_five'].unique():
    # Фильтруем данные по значению столбца is_two
    if i == False:
            profile_df[profile_df['is_two'] == True].loc[profile_df['is_five'] == i, 'avg_revenue_rub'].plot(
                kind='hist',
                bins=range(min_value, max_value+1, 50),
                alpha=0.5,
                label=f'2-4 заказа',
                legend=True
    )
            print(profile_df[profile_df['is_two'] == True].loc[profile_df['is_five'] == i, 'avg_revenue_rub'].describe())
    else:
            profile_df.loc[profile_df['is_five'] == i, 'avg_revenue_rub'].plot(
                kind='hist',
                bins=range(min_value, max_value+1, 50),
                alpha=0.5,
                label=f'5 и более заказов',
                legend=True

    )
            print(profile_df.loc[profile_df['is_five'] == i, 'avg_revenue_rub'].describe())

# Настраиваем оформление графика
plt.title('Сравнение распределения средней выручки с билета для групп пользователей 2-4 заказа и 5 и более заказов')
plt.xlabel('Выручка с билета')
plt.ylabel('Частота')
plt.legend(title='Группы пользователей по количеству заказов')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

Среднее значение в двух этих группах примерно равно. В группе с 5 и более заказами среднее значение равно 535 р, медианное равно 511 р. А в группе с 2-4 заказами среднее равно 549 р, медианное 470 р. Значения в группе с 5 и более заказами распределенны ближе к нормальному распределению.

---

**Задача 4.2.3.** Проанализируйте влияние среднего количества билетов в заказе на вероятность повторной покупки.

- Изучите распределение пользователей по среднему количеству билетов в заказе (`avg_tickets_count`) и опишите основные наблюдения.
- Разделите пользователей на несколько сегментов по среднему количеству билетов в заказе:
    - от 1 до 2 билетов;
    - от 2 до 3 билетов;
    - от 3 до 5 билетов;
    - от 5 и более билетов.
- Для каждого сегмента подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы.
- Ответьте на вопросы:
    - Как распределены пользователи по сегментам — равномерно или сконцентрировано?
    - Есть ли сегменты с аномально высокой или низкой долей повторных покупок?

---

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(7, 3))


profile_df['avg_tickets_count'].plot(
        kind='hist',
        bins=50,

    )

# Настраиваем оформление графика
plt.title('Распределение пользователей по среднему количеству билетов в заказе')
plt.xlabel('Среднее количество билетов')
plt.ylabel('Частота')

# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()



In [ ]:
profile_df['avg_tickets_count'].describe()

Данные распределены равномерно, медианное значение 2,75 почти равно среднему 2,74. Минимальное число билетов в заказе равно 1, максимальное 11.

In [ ]:
profile_df_segment_ticket_count = profile_df.copy()

In [ ]:
#напишем функцию, которая делит пользователей на сегменты в зависимости от среднего значения билетов в заказе
def create_is_avg_tickets_count_segment(row):
    if row['avg_tickets_count'] >= 1 and row['avg_tickets_count'] <= 2:
        return '1_2'
    elif row['avg_tickets_count'] > 2 and row['avg_tickets_count'] <= 3:
        return '2_3'
    elif row['avg_tickets_count'] > 3 and row['avg_tickets_count'] <= 5:
         return '3_5'
    elif row['avg_tickets_count'] > 5 :
         return '5+'
    else:
        return 'unknown'
  

In [ ]:
profile_df_segment_ticket_count['avg_tickets_count_segment'] = profile_df_segment_ticket_count.apply(create_is_avg_tickets_count_segment, axis=1)

In [ ]:
profile_df_segment_ticket_count.groupby('avg_tickets_count_segment').agg({'avg_revenue_rub':'count', 'is_two' : 'mean'})

Наибольшее число пользователей наблюдается в сегменте со средним числом билетов в заказе `2-3` - 9922 пользователя, при этом 74,2% из них совершили повторный заказ. Далее идет сегмент `1-2` билетов - 6163 пользоватлей, однако доля пользователей с повторным заказами здесь значительно меньше и составляет 40,1 %. Замыкает тройку по количеству пользователей сегмент `3-5` с количеством пользователей 5348, доля повторных составляет 62,6 %. В сегменте `5+` мало пользователей, а именно 191, доля повторных равна 32,9 %. Данных в сегменте `5+` мало по сравнению с другими сегментами.

Сегментов с аномально низкой долей повторных покупок нет.

---

#### 4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки

Изучите временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки;
- время с момента первой покупки — лайфтайм;
- средний интервал между покупками пользователей с повторными заказами.

---

**Задача 4.3.1.** Проанализируйте, как день недели, в которой была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделите день недели.
- Для каждого дня недели подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируйте.
- Ответьте на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?

---


In [ ]:
profile_df.groupby(profile_df['min_order_dt'].dt.day_name()).agg({'avg_revenue_rub':'count', 'is_two' : 'mean'}).rename(columns={'avg_revenue_rub':'count_users'})

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(13, 5))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
profile_df.groupby(profile_df['min_order_dt'].dt.day_name())['avg_revenue_rub'].count().sort_values(ascending=False).plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=90, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение количества пользователей по дням недели, в который совершена первая покупка'
)

# Настраиваем оформление графика
plt.xlabel('День недели')
plt.ylabel('Количество пользователей пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(13, 5))

# Строим столбчатую диаграмму с помощью pandas через plot(kind='bar')
profile_df.groupby(profile_df['min_order_dt'].dt.day_name())['is_two'].mean().sort_values(ascending=False).plot(
               kind='bar', # Тип графика - столбчатая диаграмма
               rot=90, # Градус вращения подписи по оси Х
               legend=False, # Выключаем легенду
               title=f'Распределение доли пользователей,совершивших повторный заказ, по дням недели, в который совершена первая покупка'
)

# Настраиваем оформление графика
plt.xlabel('День недели')
plt.ylabel('Доля пользователей пользователей')
# Добавляем сетку графика
plt.grid()

# Выводим график
plt.show()

Большинство пользователей совершили первый заказ в субботу. Меньше всего первых заказов было создано в воскресенье.

Доли повторных заказов по дням неделям находятся примерно на одном уровне, можно сделать вывод, что день недели, в которую совершена первая покупка, не влияет на вероятность возврата клиента.

---

**Задача 4.3.2.** Изучите, как средний интервал между заказами влияет на удержание клиентов.

- Рассчитайте среднее время между заказами для двух групп пользователей:
    - совершившие 2–4 заказа;
    - совершившие 5 и более заказов.
- Исследуйте, как средний интервал между заказами влияет на вероятность повторного заказа, и сделайте выводы.

---


In [ ]:
for i in profile_df['is_five'].unique():
    # Фильтруем данные по значению столбца is_two
    if i == False:
            print('Группа 2-4 заказа:')
            print(profile_df[profile_df['is_two'] == True].loc[profile_df['is_five'] == i, 'avg_time_between_orders'].describe())
    else:
            print('Группа 5+ заказов:')
            print(profile_df.loc[profile_df['is_five'] == i, 'avg_time_between_orders'].describe())


Среднее время между заказами в сегменте `2-4 заказа` равно 21 день 9 часов, а медианное 9 дней 3 часа, что говорит о неравномерном распределении данных в этом сегменте. 

Среднее время между заказами в сегменте `5+ заказов` равно 9 дней 9 часов, а медианное 8 дней 5 часов, данные распределены более менее равномерно.

То есть время между заказами с их увеличением уменьшилось, но в среднем всего лишь на день, таким образом, сохранился примерно тот же уровень.


---

#### 4.4. Корреляционный анализ количества покупок и признаков пользователя

Изучите, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используйте универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.

---

**Задача 4.4.1:** Проведите корреляционный анализ:
- Рассчитайте коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов (`total_orders`). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируйте полученные результаты. Если полученные значения будут близки к нулю, проверьте разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - от 2 до 4 заказов;
    - от 5 и выше.
- Визуализируйте результат корреляции с помощью тепловой карты.
- Ответьте на вопрос: какие признаки наиболее связаны с количеством заказов?

---

In [ ]:
# Вычисляем корреляционную матрицу с использованием phi_k, у меня total_orders называется count_orders
correlation_matrix = profile_df[['min_order_dt', 'first_device_type_canonical', 'first_region_name', 'first_service_name', 'first_event_type_main',
                         'avg_revenue_rub', 'avg_tickets_count', 'count_orders']].phik_matrix(interval_cols=['avg_revenue_rub', 'avg_tickets_count', 'count_orders'])

# Выводим результат
print('Корреляционная матрица с коэффициентом phi_k для переменной count_orders')
correlation_matrix.loc[correlation_matrix.index != 'count_orders'][['count_orders']].sort_values(by='count_orders', ascending=False)

Наибольшая корреляция количества заказов наблюдается с датой первого заказа `min_order_dt` - 0.43, средним количеством билетов `avg_tickets_count` - 0,23 и средней выручкой `avg_revenue_rub` - 0.22.  

Построим тепловую карту с корреляцией.

In [ ]:
# Строим тепловую карту
plt.figure(figsize=(2, 6))

data_heatmap = correlation_matrix.loc[correlation_matrix.index != 'count_orders'][['count_orders']].sort_values(by='count_orders', ascending=False)
sns.heatmap(data_heatmap,
            annot=True, # Отображаем численные значения в ячейках карты
            fmt='.2f', # Форматируем значения корреляции: два знака после точки
            cmap='coolwarm', # Устанавливаем цветовую гамму от красного (макс. значение) к синему
            linewidths=0.5, # Форматируем линию между ячейками карты
            cbar=False # Отключаем цветовую шкалу,
            
           )

# Добавляем заголовок и подпись по оси Х
plt.title('Тепловая карта коэффициента phi_k \n для данных count_orders')
plt.xlabel('Число заказов')

# Выводим график
plt.show()

### 5. Общий вывод и рекомендации

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.

В процессе работы проводился анализ лояльности пользователей Яндекс Афиши с помощью Python.

Выгрузка из базы данных SQL позволила собрать следующие данные:

`user_id` — уникальный идентификатор пользователя, совершившего заказ;

`device_type_canonical` — тип устройства, с которого был оформлен заказ ( mobile — мобильные устройства, desktop — стационарные);

`order_id` — уникальный идентификатор заказа;

`order_dt` — дата создания заказа (используйте данные created_dt_msk );

`order_ts` — дата и время создания заказа (используйте данные created_ts_msk );

`currency_code` — валюта оплаты;

`revenue` — выручка от заказа;

`tickets_count` — количество купленных билетов;

`days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;

`event_id` — уникальный идентификатор мероприятия;

`service_name` — название билетного оператора;

`event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);

`region_name` — название региона, в котором прошло мероприятие;

`city_name` — название города, в котором прошло мероприятие.

В результате предобработки данных были выполнены следующие действия:

Вся выручка приведена к единой валюте - рублю, округляя до двух знаков после запятой, в соответствии с курсом тенге в конкретную дату. Данные были сохранены в столбце revenue_rub.

Изучены пропуски в данных. Пропуски присутствуют только в столбце days_since_days, что и должно быть, так как в этом столбце хранится количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено. Пропуски оставлены без изменений.

Изучены типы данных. Оптимизирован тип данных float в столбцах revenue,days_since_prev, revenue_rub и тип данных int в столбцах tickets_count,event_id.

Изучены ключевые значения в столбцах. В категориальных данных все значения уникальны, значений обозначающих пропуски в них нет, нормализация данных не требуется.

Данные проверили на явные и неявные дубликаты. Было найдено 31 явных дубликатов, они были удалены. Неявные дубликаты отсутствуют.

Проверено распределение численных данных и наличие в них выбросов. Данные в столбце revenue_rub распределены неравномерно с правым хвостом, среднее значение 555.18 сильно отличается от медианного 351.14. Это происходит из-за выбросов, к таковым в данном случае относятся значения, превыщающие 1835 рублей. Значения в этом столбце были отфильтрованы по 99 перцентилю.

Данные в столбце tickets_count также распределены неравномерно, наблюдается правый хвост, к выбросам в данном случае относятся значения, превыщающие 7. В среднем покупают 3 билета.


Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?

Сформирован датафрейм, описывающий профиль пользователя.

В выборке число пользователей составляет 21844. Cредняя выручка с одного заказа составляет 542.75 рубля. Доля пользователей, совершивших 2 и более заказа, составляет 0.62, то есть почти 2/3 всех пользователей совершили 2 и более заказа. Доля пользователей, совершивших 5 и более заказов, составляет 0.29, то есть почти 1/3 всех пользователей совершила 5 и более заказов.

Из статистических показателей по общему числу заказов видно, что минимальное число заказов составляет 1, максимальное 10181, среднее 13,1, а медианное 2, то есть данные распределены неравномерно, из-за выбросов искажено среднее значение. Огромные числа, такое как максимальное значение, можно отнести к аномальным значениям, и отфильтровать по 99 процентилю, что и было сделано.

Из статистических показателей по среднему числу билетов в заказе видно, что минимальное число билетов в заказе составляет 1, максимальное 11, среднее 2,75 и медианное 2,75 , то есть данные распределены равномерно.

Из статистических показателей по среднему количеству дней между покупками видно, что минимальное количество дней между покупками (если заказ не один, а таких 13482 записи) составляет 1 секунду, максимальное 148 дней, среднее почти 16 дней, а медианное чуть больше 8 дней , что говорит о том, что данные распределены неравномерно. Но большие значения между заказами вполне могут быть в реальности, поэтому их не стоит относить к аномальным.

Какие признаки первого заказа связаны с возвратом пользователей?

В сегментах по типу первого мероприятия показатели находятся примерно на одном уровне. В категории выставки 64% пользователей, совершили 2 и более заказа, далее идет категория театр (63,4%) и категория концерты (61,8%). Однако в сегменте выставки мало пользователей, и доля может быть недостоверной. Про остальные две категории можно сказать, что пользователи в них чаще возвращаются на Яндекс Афишу.

В сегментах по типу устройства с которого совершена первая покупка  примерно 60% пользователей в каждом сегменте совершают 2 и более заказа. При этом данных в каждом сегменте достаточно, хотя данные в этих сегментах распределены неравномерно (82,8 % пользователей совершали первый заказ с мобильного телефона, а 17,1 % с компьютера).

В сегментах по региону проведения первого мероприятия в Верхнеозерском крае 100% пользователей совершили 2 и более заказов (а это потому что там вообще 1 пользователь). В Озернопольской области 89,6 % пользователей (здесь их тоже очень мало),а в Радужнопольском крае 78,2% (и здесь их очень мало). Пятерки регионов, лидирующих по количеству пользователей, здесь нет.

В сегментах по билетному оператору, продавшему билеты на первый заказ, можно отметить только операторов Край билетов c количеством пользователей 454 и Дом культуры с количеством пользователей 353. Доли пользователей, совершивших 2 и более заказов, равны около 65 %.


Гипотеза: Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты, оказалась неверна. Доля пользователей, совершивших 2 и более заказов, у которых первый заказ был на спортивные мероприятия составляет 55,7 %, а у которых первый заказ был на концерт составляет 61,8 %. При этом стоит отметить, что пользователей, совершивших первый заказ на концерт гораздо больше, чем тех, что относятся к сегменту спорт.

Гипотеза: В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах, оказалась верна для тех данных, где количество пользователей не слишком мало. У первой тройки по количеству пользователей Каменевский регион, Североярская область и Широковская область доля тех, кто совершил 2 и более заказов, составляет около 63 %, что в целом больше, остальных. Хотя встречаются регионы с меньшим количеством пользователей, но с долей равной или немного превышающей это значение.


Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?

Распределение средней выручки у тех, кто совершил 2 и более заказа, сдвинуто в правую область по отношению к тем, кто совершил один заказ: в среднем выручка у пользователей, которые возвращаются 400-600 рублей. А у пользователей совершивших один заказ показатели выручки распределены неравномерно, с правым хвостом, медианное значение 377 р.
Среднее значение выручки в группах 2-4 заказа и 5+ заказов примерно равно. В группе с 5 и более заказами среднее значение равно 535 р, медианное равно 511 р. А в группе с 2-4 заказами среднее равно 549 р, медианное 470 р.

Наибольшее число пользователей наблюдается в сегменте со средним числом билетов в заказе 2-3 - 9922 пользователя, при этом 74,2% из них совершили повторный заказ. Далее идет сегмент 1-2 билетов - 6163 пользоватлей, однако доля пользователей с повторным заказами здесь значительно меньше и составляет 40,1 %. Замыкает тройку по количеству пользователей сегмент 3-5 с количеством пользователей 5348, доля повторных составляет 62,6 %. В сегменте 5+ мало пользователей, а именно 191, доля повторных равна 32,9 %. Данных в сегменте 5+ мало по сравнению с другими сегментами.


Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?

Большинство пользователей совершили первый заказ в субботу. Меньше всего первых заказов было создано в воскресенье.
Доли повторных заказов по дням неделям находятся примерно на одном уровне, можно сделать вывод, что день недели, в которую совершена первая покупка, не влияет на вероятность возврата клиента.

Среднее время между заказами в сегменте 2-4 заказа равно 21 день 9 часов, а медианное 9 дней 3 часа, что говорит о неравномерном распределении данных в этом сегменте.Среднее время между заказами в сегменте 5+ заказов равно 9 дней 9 часов, а медианное 8 дней 5 часов, данные распределены более менее равномерно.То есть время между заказами с их увеличением уменьшилось, но в среднем всего лишь на день, таким образом, сохранился примерно тот же уровень.


Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?

Наибольшая корреляция количества заказов наблюдается с датой первого заказа min_order_dt - 0.43, средним количеством билетов avg_tickets_count - 0,23 и средней выручкой avg_revenue_rub - 0.22. 

Стоит обратить внимание на сегменты по региону проведения первого мероприятия и по билетному оператору, продавшему билеты на первый заказ, с низкой долей пользователей, совершивших повторный заказ, а также увеличить маркетинговую кампанию в дни недели с наименьшим количеством заказов.

### 6. Финализация проекта и публикация в Git

Когда вы закончите анализировать данные, оформите проект, а затем опубликуйте его.

Выполните следующие действия:

1. Создайте файл `.gitignore`. Добавьте в него все временные и чувствительные файлы, которые не должны попасть в репозиторий.
2. Сформируйте файл `requirements.txt`. Зафиксируйте все библиотеки, которые вы использовали в проекте.
3. Вынести все чувствительные данные (параметры подключения к базе) в `.env`файл.
4. Проверьте, что проект запускается и воспроизводим.
5. Загрузите проект в публичный репозиторий — например, на GitHub. Убедитесь, что все нужные файлы находятся в репозитории, исключая те, что в `.gitignore`. Ссылка на репозиторий понадобится для отправки проекта на проверку. Вставьте её в шаблон проекта в тетрадке Jupyter Notebook перед отправкой проекта на ревью.

**Вставьте ссылку на проект в этой ячейке тетрадки перед отправкой проекта на ревью.**